In [ ]:
import os
import shutil
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_classic.chains.question_answering import load_qa_chain


In [ ]:
load_dotenv(find_dotenv(usecwd=True), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY não encontrada no .env"


In [ ]:
embeddings_models = OpenAIEmbeddings()
llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=2000)


In [ ]:
PDF_PATH = "../os-sertoes.pdf"
PERSIST_DIR = "text_index"
REBUILD = True  # deixe True só na primeira execução; depois mude para False

def limpar(texto):
    linhas = [
        l for l in texto.splitlines()
        if "b i b l i" not in l and ".indd" not in l
    ]
    return "\n".join(linhas).strip()

if REBUILD or not os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR, ignore_errors=True)

    pages = PyMuPDFLoader(PDF_PATH).load()
    for p in pages:
        p.page_content = limpar(p.page_content)
    pages = [p for p in pages if len(p.page_content) > 300]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
        add_start_index=True,
    )
    chunks = splitter.split_documents(pages)
    print(len(pages), "páginas |", len(chunks), "chunks")

    vectordb = Chroma.from_documents(chunks, embedding=embeddings_models, persist_directory=PERSIST_DIR)
else:
    vectordb = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings_models)


AttributeError: 'Chroma' object has no attribute 'persist'

In [ ]:
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20},
)

chain = load_qa_chain(llm, chain_type="stuff")


In [ ]:
def ask(question, mostrar_fontes=True):
    docs = retriever.invoke(question)
    if mostrar_fontes:
        for d in docs:
            print(f"[pág. {d.metadata.get('page')}] {d.page_content[:200]!r}\n")
    result = chain.invoke({"input_documents": docs, "question": question})
    return result["output_text"]

In [51]:
user_question = input("Question: ")
print("answer:", ask(user_question))
# print("answer: ", answer)

answer: No livro "Os sertões", Euclides da Cunha menciona Antônio Vicente Mendes Maciel, conhecido como Antônio Conselheiro, como uma figura central na Guerra de Canudos. Ele descreve Conselheiro como um líder religioso que, nascido em Quixeramobim, no Ceará, percorreu o sertão da Bahia construindo igrejas e reformando cemitérios antes de formar a comunidade de Belo Monte, nas margens do Rio Vaza-Barris, em Canudos. 

Euclides da Cunha destaca que a comunidade de Canudos foi vista como uma ameaça pelo governo, que alegava que a liderança de Conselheiro representava uma reação monárquica que poderia minar as instituições republicanas. Assim, Conselheiro é apresentado não apenas como um líder espiritual, mas também como um símbolo de resistência e de uma complexa interação entre religiosidade e política no contexto da época. 

A sua figura é associada à resistência dos habitantes de Canudos contra as expedições militares que buscavam destruir a comunidade, mostrando a disparidade entre a